# 🏠 سامانه هوشمند پیش‌بینی قیمت مسکن تهران

**پروژه درس پروژه نرم‌افزار**

در این Notebook مراحل کامل پروژه به‌ترتیب انجام می‌شود:

1. نصب و وارد کردن کتابخانه‌ها
2. بارگذاری دیتاست
3. بررسی اولیه داده‌ها
4. پاک‌سازی و پیش‌پردازش
5. آماده‌سازی ویژگی‌ها
6. تقسیم داده‌ها به آموزش و آزمون
7. آموزش Linear Regression
8. آموزش Random Forest Regression
9. مقایسه مدل‌ها
10. بررسی نموداری مدل نهایی
11. پیش‌بینی یک خانه جدید
12. ذخیره مدل
13. ساخت رابط کاربری با Gradio

> **راهنما:** در Google Colab از منوی `Runtime > Run all` استفاده کنید. اگر فایل CSV داخل محیط Colab نباشد، Notebook از شما می‌خواهد آن را آپلود کنید.

## 1) نصب کتابخانه‌های موردنیاز

In [ ]:
!pip install -q "gradio>=6,<7" joblib

## 2) وارد کردن کتابخانه‌ها

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import gradio as gr

print("کتابخانه‌ها با موفقیت بارگذاری شدند ✅")

## 3) بارگذاری دیتاست

فایل اصلی پروژه:

`1632300362534233.csv`

اگر فایل از قبل در `/content` موجود نباشد، پنجره آپلود نمایش داده می‌شود.

In [ ]:
csv_path = Path("/content/1632300362534233.csv")

if not csv_path.exists():
    from google.colab import files

    print("فایل CSV را انتخاب و آپلود کنید.")
    uploaded = files.upload()

    if len(uploaded) == 0:
        raise FileNotFoundError("هیچ فایلی آپلود نشد.")

    csv_path = Path("/content") / next(iter(uploaded.keys()))

df = pd.read_csv(csv_path)

print("دیتاست با موفقیت بارگذاری شد ✅")
print("ابعاد اولیه دیتاست:", df.shape)

df.head()

## 4) بررسی اولیه داده‌ها

In [ ]:
print("نام ستون‌ها:")
print(df.columns.tolist())

print("\nتعداد مقادیر خالی:")
print(df.isnull().sum())

print("\nاطلاعات دیتاست:")
df.info()

In [ ]:
df.describe(include="all")

## 5) پاک‌سازی و پیش‌پردازش داده‌ها

در این مرحله:

- ستون `Area` از متن به عدد تبدیل می‌شود.
- مقادیر خالی `Address` حذف می‌شوند.
- داده‌های غیرقابل‌تبدیل متراژ حذف می‌شوند.
- متراژهای بیشتر از 1000 متر از این پروژه کنار گذاشته می‌شوند.
- ستون‌های بولی به 0 و 1 تبدیل می‌شوند.

In [ ]:
# تبدیل Area از متن به عدد
df["Area"] = (
    df["Area"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

df["Area"] = pd.to_numeric(
    df["Area"],
    errors="coerce"
)

# حذف مقادیر خالی ضروری
df = df.dropna(
    subset=["Area", "Address"]
).copy()

# حذف متراژهای غیرعادی برای این پروژه
df = df[
    (df["Area"] > 0) &
    (df["Area"] <= 1000)
].copy()

# تبدیل True/False به 1/0
binary_columns = [
    "Parking",
    "Warehouse",
    "Elevator"
]

for column in binary_columns:
    df[column] = df[column].astype(int)

# مرتب‌سازی مجدد ایندکس‌ها
df.reset_index(
    drop=True,
    inplace=True
)

print("پاک‌سازی داده‌ها انجام شد ✅")
print("ابعاد دیتاست پس از پاک‌سازی:", df.shape)
print("تعداد محله‌ها:", df["Address"].nunique())

In [ ]:
df.head()

### بررسی آماری متراژ پس از پاک‌سازی

In [ ]:
df["Area"].describe()

## 6) آماده‌سازی داده‌ها برای مدل

ستون `Price` متغیر هدف است.

ستون `Price(USD)` از ورودی مدل حذف می‌شود، زیرا از قیمت اصلی محاسبه شده و نباید به‌عنوان ویژگی ورودی استفاده شود.

برای ستون `Address` از **One-Hot Encoding** استفاده می‌کنیم.

In [ ]:
# حذف Price(USD)
df_model = df.drop(
    columns=["Price(USD)"]
).copy()

# تبدیل Address به ویژگی‌های عددی
df_model = pd.get_dummies(
    df_model,
    columns=["Address"],
    drop_first=True,
    dtype=int
)

# ورودی‌ها و خروجی مدل
X = df_model.drop(
    columns=["Price"]
)

y = df_model["Price"]

print("ابعاد X:", X.shape)
print("ابعاد y:", y.shape)

## 7) تقسیم داده‌ها به Train و Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

## 8) آموزش مدل Linear Regression

In [ ]:
linear_model = LinearRegression()

linear_model.fit(
    X_train,
    y_train
)

linear_pred = linear_model.predict(
    X_test
)

linear_mae = mean_absolute_error(
    y_test,
    linear_pred
)

linear_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        linear_pred
    )
)

linear_r2 = r2_score(
    y_test,
    linear_pred
)

print("Linear Regression")
print(f"MAE : {linear_mae:,.0f} تومان")
print(f"RMSE: {linear_rmse:,.0f} تومان")
print(f"R²  : {linear_r2:.4f}")

## 9) آموزش مدل Random Forest Regression

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

rf_pred = rf_model.predict(
    X_test
)

rf_mae = mean_absolute_error(
    y_test,
    rf_pred
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_pred
    )
)

rf_r2 = r2_score(
    y_test,
    rf_pred
)

print("Random Forest Regression")
print(f"MAE : {rf_mae:,.0f} تومان")
print(f"RMSE: {rf_rmse:,.0f} تومان")
print(f"R²  : {rf_r2:.4f}")

## 10) مقایسه دو مدل

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest"
    ],
    "MAE": [
        linear_mae,
        rf_mae
    ],
    "RMSE": [
        linear_rmse,
        rf_rmse
    ],
    "R2": [
        linear_r2,
        rf_r2
    ]
})

results

### نتیجه انتخاب مدل

مدلی مناسب‌تر است که:

- **MAE کمتر** داشته باشد.
- **RMSE کمتر** داشته باشد.
- **R² بالاتر** داشته باشد.

در این دیتاست، Random Forest عملکرد بهتری دارد و به‌عنوان مدل نهایی پروژه استفاده می‌شود.

## 11) نمودار قیمت واقعی در برابر قیمت پیش‌بینی‌شده

In [ ]:
plt.figure(
    figsize=(8, 6)
)

plt.scatter(
    y_test,
    rf_pred,
    alpha=0.5
)

min_price = min(
    y_test.min(),
    rf_pred.min()
)

max_price = max(
    y_test.max(),
    rf_pred.max()
)

plt.plot(
    [min_price, max_price],
    [min_price, max_price]
)

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title(
    "Random Forest - Actual vs Predicted Prices"
)

plt.tight_layout()
plt.show()

## 12) بررسی چند پیش‌بینی مدل

In [ ]:
comparison = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": rf_pred
})

comparison["Error"] = (
    comparison["Actual Price"] -
    comparison["Predicted Price"]
).abs()

comparison.head(10)

### 10 نمونه دارای بیشترین خطا

In [ ]:
comparison.sort_values(
    by="Error",
    ascending=False
).head(10)

## 13) پیش‌بینی قیمت یک خانه جدید

نمونه:

- متراژ: 120 متر
- تعداد اتاق: 2
- پارکینگ: دارد
- انباری: دارد
- آسانسور: دارد
- محله: Shahran

In [ ]:
new_house = pd.DataFrame(
    0,
    index=[0],
    columns=X.columns
)

new_house["Area"] = 120
new_house["Room"] = 2
new_house["Parking"] = 1
new_house["Warehouse"] = 1
new_house["Elevator"] = 1

sample_address = "Shahran"
address_column = "Address_" + sample_address

# اگر این محله ستون اختصاصی داشته باشد
if address_column in new_house.columns:
    new_house[address_column] = 1

sample_prediction = rf_model.predict(
    new_house
)[0]

print(
    f"قیمت پیش‌بینی‌شده: "
    f"{sample_prediction:,.0f} تومان"
)

print(
    f"حدود "
    f"{sample_prediction / 1_000_000_000:.2f} "
    f"میلیارد تومان"
)

## 14) ذخیره مدل نهایی

In [ ]:
model_data = {
    "model": rf_model,
    "columns": X.columns.tolist(),
    "addresses": sorted(
        df["Address"].unique()
    )
}

joblib.dump(
    model_data,
    "tehran_house_price_model.pkl"
)

print(
    "مدل با موفقیت در فایل "
    "tehran_house_price_model.pkl "
    "ذخیره شد ✅"
)

# 15) رابط کاربری نرم‌افزار با Gradio

این بخش رابط نهایی پروژه است.

کاربر می‌تواند موارد زیر را وارد کند:

- متراژ
- تعداد اتاق
- محله
- پارکینگ
- انباری
- آسانسور

برای جلوگیری از خطا، ورودی‌ها اعتبارسنجی می‌شوند.

In [ ]:
addresses = sorted(
    df["Address"].unique()
)


def message_html(message, icon="⚠️"):
    return f"""
    <div dir="rtl"
         style="text-align:center; padding:15px;">
        <h3>{icon} {message}</h3>
    </div>
    """


def predict_house_price(
    area,
    room,
    parking,
    warehouse,
    elevator,
    address
):
    try:
        # بررسی ورودی‌های خالی
        if area is None:
            return message_html(
                "لطفاً متراژ خانه را وارد کنید."
            )

        if room is None:
            return message_html(
                "لطفاً تعداد اتاق را وارد کنید."
            )

        if address is None or str(address).strip() == "":
            return message_html(
                "لطفاً محله را انتخاب کنید."
            )

        # تبدیل نوع داده‌ها
        area = float(area)
        room = float(room)

        # بررسی معتبر بودن اعداد
        if not np.isfinite(area):
            return message_html(
                "متراژ واردشده معتبر نیست."
            )

        if not np.isfinite(room):
            return message_html(
                "تعداد اتاق واردشده معتبر نیست."
            )

        # بررسی محدوده متراژ
        if area < 30:
            return message_html(
                "متراژ خانه نمی‌تواند کمتر از ۳۰ متر باشد."
            )

        if area > 1000:
            return message_html(
                "حداکثر متراژ قابل قبول ۱۰۰۰ متر است."
            )

        # بررسی تعداد اتاق
        if room < 0:
            return message_html(
                "تعداد اتاق نمی‌تواند منفی باشد."
            )

        if room > 10:
            return message_html(
                "حداکثر تعداد اتاق قابل قبول ۱۰ است."
            )

        if room != int(room):
            return message_html(
                "تعداد اتاق باید عدد صحیح باشد."
            )

        area = int(area)
        room = int(room)

        # بررسی محله
        if address not in addresses:
            return message_html(
                "محله انتخاب‌شده در اطلاعات پروژه وجود ندارد."
            )

        # ساخت رکورد خانه جدید
        new_house = pd.DataFrame(
            0,
            index=[0],
            columns=X.columns
        )

        new_house["Area"] = area
        new_house["Room"] = room
        new_house["Parking"] = 1 if parking else 0
        new_house["Warehouse"] = 1 if warehouse else 0
        new_house["Elevator"] = 1 if elevator else 0

        # One-Hot مربوط به محله
        address_column = "Address_" + address

        if address_column in new_house.columns:
            new_house[address_column] = 1

        # پیش‌بینی
        predicted_price = rf_model.predict(
            new_house
        )[0]

        if predicted_price < 0:
            return message_html(
                "امکان ارائه پیش‌بینی معتبر برای این مشخصات وجود ندارد."
            )

        return f"""
        <div dir="rtl"
             style="text-align:center; padding:15px;">

            <h3>قیمت تقریبی ملک:</h3>

            <h2>
                {predicted_price:,.0f} تومان
            </h2>

            <h3>
                حدود
                {predicted_price / 1_000_000_000:.2f}
                میلیارد تومان
            </h3>

        </div>
        """

    except Exception:
        return message_html(
            "خطایی در انجام پیش‌بینی رخ داد. "
            "لطفاً اطلاعات واردشده را بررسی کنید.",
            icon="❌"
        )


css = """
.gradio-container {
    direction: rtl;
    max-width: 1100px !important;
    margin: auto !important;
}

#title {
    text-align: center;
    margin-bottom: 20px;
}

#predict-btn {
    font-size: 18px !important;
    font-weight: bold !important;
}

#result-box {
    text-align: center;
    font-size: 18px;
}
"""


with gr.Blocks(
    title="سامانه هوشمند پیش‌بینی قیمت مسکن تهران"
) as app:

    gr.Markdown(
        """
        # 🏠 سامانه هوشمند پیش‌بینی قیمت مسکن تهران

        مشخصات ملک موردنظر را وارد کرده و سپس
        روی دکمه **پیش‌بینی قیمت** کلیک کنید.
        """,
        elem_id="title"
    )

    with gr.Row():

        area_input = gr.Number(
            label="📐 متراژ خانه",
            value=100,
            minimum=30,
            maximum=1000,
            precision=0
        )

        room_input = gr.Number(
            label="🛏 تعداد اتاق",
            value=2,
            minimum=0,
            maximum=10,
            precision=0
        )

    address_input = gr.Dropdown(
        choices=addresses,
        value=(
            "Punak"
            if "Punak" in addresses
            else addresses[0]
        ),
        label="📍 محله تهران"
    )

    with gr.Row():

        parking_input = gr.Checkbox(
            label="🚗 پارکینگ",
            value=True
        )

        warehouse_input = gr.Checkbox(
            label="📦 انباری",
            value=True
        )

        elevator_input = gr.Checkbox(
            label="⚙️ آسانسور",
            value=True
        )

    predict_button = gr.Button(
        "🔍 پیش‌بینی قیمت ملک",
        variant="primary",
        elem_id="predict-btn"
    )

    output = gr.HTML(
        label="💰 نتیجه پیش‌بینی",
        elem_id="result-box"
    )

    gr.HTML(
        f"""
        <hr>
        <div dir="rtl"
             style="text-align:right;">

            <b>مدل مورد استفاده:</b>
            <span dir="ltr">
                Random Forest Regression
            </span>

            <br>

            <b>تعداد داده‌های مورد استفاده:</b>
            {len(df)} ملک

        </div>
        """
    )

    predict_button.click(
        fn=predict_house_price,
        inputs=[
            area_input,
            room_input,
            parking_input,
            warehouse_input,
            elevator_input,
            address_input
        ],
        outputs=output
    )

print("رابط نرم‌افزار ساخته شد ✅")

## 16) اجرای نرم‌افزار

با اجرای سلول زیر، نرم‌افزار اجرا می‌شود و در Google Colab لینک عمومی Gradio نیز ساخته می‌شود.

In [ ]:
app.launch(
    share=True,
    css=css
)

# ✅ پایان پروژه

در پایان پروژه:

- داده‌ها پاک‌سازی شدند.
- دو مدل رگرسیون آموزش داده شد.
- عملکرد مدل‌ها مقایسه شد.
- Random Forest به‌عنوان مدل نهایی انتخاب شد.
- امکان پیش‌بینی خانه جدید ایجاد شد.
- مدل ذخیره شد.
- رابط کاربری تحت وب ساخته شد.
- ورودی‌های نرم‌افزار اعتبارسنجی شدند.

**عنوان پیشنهادی برای تحویل:**

> طراحی و پیاده‌سازی سامانه هوشمند پیش‌بینی قیمت مسکن در تهران با استفاده از الگوریتم‌های یادگیری ماشین